In [1]:
# Import libraries

from pathlib import Path
import pickle

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
import faiss

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 150)

In [2]:
# Define project paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PREPROCESSED_DIR = PROJECT_ROOT / "data" / "02_preprocessed"
RAG_DOCS_DIR = PREPROCESSED_DIR / "rag_documents"
VECTOR_DB_DIR = PROJECT_ROOT / "vector_db"

RAG_DOCS_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PREPROCESSED_DIR:", PREPROCESSED_DIR)
print("RAG_DOCS_DIR:", RAG_DOCS_DIR)
print("VECTOR_DB_DIR:", VECTOR_DB_DIR)

PROJECT_ROOT: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform
PREPROCESSED_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\02_preprocessed
RAG_DOCS_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\02_preprocessed\rag_documents
VECTOR_DB_DIR: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\vector_db


In [3]:
# Load RAG-ready CMS plan data from Notebook 1

input_path = PREPROCESSED_DIR / "rag_ready_plan_data_sample.csv"

if not input_path.exists():
    raise FileNotFoundError(
        f"Missing file: {input_path}\n"
        "Run Notebook 1 first and make sure rag_ready_plan_data_sample.csv was created."
    )

plan_df = pd.read_csv(input_path)

print("Shape:", plan_df.shape)
display(plan_df.head())

Shape: (1000, 12)


,plan_id,issuer_id,issuer_name,state,plan_name,metal_level,plan_type,hsa_eligible,deductible_integrated,oop_integrated,out_of_pocket_individual,rag_text
0,21989AK0030001,21989,Delta Dental of Alaska,AK,Delta Dental Premier Plan,Low,Indemnity,NaN,NaN,NaN,NaN,Plan Name: Delta Dental Premier Plan\nPlan ID: 21989AK0030001\nIssuer: Delta Dental of Alaska\nState: AK\nMetal Level: Low\nPlan Type: Indemnity\n...
1,21989AK0050001,21989,Delta Dental of Alaska,AK,Delta Dental PPO 1000 Plan,High,PPO,NaN,NaN,NaN,450.0,Plan Name: Delta Dental PPO 1000 Plan\nPlan ID: 21989AK0050001\nIssuer: Delta Dental of Alaska\nState: AK\nMetal Level: High\nPlan Type: PPO\nHSA ...
2,21989AK0050002,21989,Delta Dental of Alaska,AK,Delta Dental PPO 1500 Plan,High,PPO,NaN,NaN,NaN,450.0,Plan Name: Delta Dental PPO 1500 Plan\nPlan ID: 21989AK0050002\nIssuer: Delta Dental of Alaska\nState: AK\nMetal Level: High\nPlan Type: PPO\nHSA ...
3,21989AK0070001,21989,Delta Dental of Alaska,AK,Delta Dental Premier Healthy Smiles,Low,Indemnity,NaN,NaN,NaN,NaN,Plan Name: Delta Dental Premier Healthy Smiles\nPlan ID: 21989AK0070001\nIssuer: Delta Dental of Alaska\nState: AK\nMetal Level: Low\nPlan Type: I...
4,21989AK0080001,21989,Delta Dental of Alaska,AK,"Delta Dental Premier 1000, 100/80/50, 50",High,Indemnity,NaN,NaN,NaN,NaN,"Plan Name: Delta Dental Premier 1000, 100/80/50, 50\nPlan ID: 21989AK0080001\nIssuer: Delta Dental of Alaska\nState: AK\nMetal Level: High\nPlan T..."


In [4]:
# Check missing values

missing_summary = plan_df.isna().sum().reset_index()
missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(plan_df) * 100).round(2)

display(missing_summary.sort_values("missing_pct", ascending=False))

,column,missing_count,missing_pct
10,out_of_pocket_individual,858,85.8
9,oop_integrated,194,19.4
7,hsa_eligible,194,19.4
8,deductible_integrated,194,19.4
3,state,0,0.0
2,issuer_name,0,0.0
1,issuer_id,0,0.0
0,plan_id,0,0.0
6,plan_type,0,0.0
5,metal_level,0,0.0


In [5]:
# Confirm RAG text exists

if "rag_text" not in plan_df.columns:
    raise ValueError("The column 'rag_text' is missing. Recheck Notebook 1 output.")

print(plan_df["rag_text"].iloc[0])

Plan Name: Delta Dental Premier Plan
Plan ID: 21989AK0030001
Issuer: Delta Dental of Alaska
State: AK
Metal Level: Low
Plan Type: Indemnity
HSA Eligible: Not available

Deductible Design:
Medical and drug deductibles integrated: Not available.

Out-of-Pocket Design:
Medical and drug maximum out-of-pocket integrated: Not available.
The individual in-network tier 1 maximum out-of-pocket amount is Not available.

Benefits Context:
This plan record comes from public CMS Exchange Public Use File data.
It can be used for benefits search, plan comparison, document retrieval, and RAG question answering.


In [6]:
# Clean old generated RAG documents before creating new ones

for old_file in RAG_DOCS_DIR.glob("*.txt"):
    old_file.unlink()

print("Old RAG text files removed.")

Old RAG text files removed.


In [7]:
# Create one text document per CMS plan record

for idx, row in plan_df.iterrows():
    plan_id = (
        str(row["plan_id"])
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(":", "_")
    )
    
    file_name = f"plan_{idx:04d}_{plan_id}.txt"
    file_path = RAG_DOCS_DIR / file_name

    file_path.write_text(str(row["rag_text"]), encoding="utf-8")

print(f"Created {len(plan_df)} plan documents in {RAG_DOCS_DIR}")

Created 1000 plan documents in c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\02_preprocessed\rag_documents


In [8]:
# Preview generated text files

text_files = sorted(RAG_DOCS_DIR.glob("*.txt"))

print("Number of text files:", len(text_files))

if len(text_files) == 0:
    raise FileNotFoundError("No RAG text files were created.")

print("First file:", text_files[0].name)
print(text_files[0].read_text(encoding="utf-8"))

Number of text files: 1000
First file: plan_0000_21989AK0030001.txt
Plan Name: Delta Dental Premier Plan
Plan ID: 21989AK0030001
Issuer: Delta Dental of Alaska
State: AK
Metal Level: Low
Plan Type: Indemnity
HSA Eligible: Not available

Deductible Design:
Medical and drug deductibles integrated: Not available.

Out-of-Pocket Design:
Medical and drug maximum out-of-pocket integrated: Not available.
The individual in-network tier 1 maximum out-of-pocket amount is Not available.

Benefits Context:
This plan record comes from public CMS Exchange Public Use File data.
It can be used for benefits search, plan comparison, document retrieval, and RAG question answering.


In [9]:
# Create simple text chunking function

def chunk_text(text, chunk_size=700, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [10]:
# Chunk all plan documents

chunk_records = []

for file_path in text_files:
    text = file_path.read_text(encoding="utf-8")
    chunks = chunk_text(text)

    for chunk_idx, chunk in enumerate(chunks):
        chunk_records.append({
            "document_name": file_path.name,
            "chunk_id": f"{file_path.stem}_chunk_{chunk_idx}",
            "text": chunk,
        })

chunks_df = pd.DataFrame(chunk_records)

print("Chunks created:", len(chunks_df))
display(chunks_df.head())

Chunks created: 1391


,document_name,chunk_id,text
0,plan_0000_21989AK0030001.txt,plan_0000_21989AK0030001_chunk_0,Plan Name: Delta Dental Premier Plan\nPlan ID: 21989AK0030001\nIssuer: Delta Dental of Alaska\nState: AK\nMetal Level: Low\nPlan Type: Indemnity\n...
1,plan_0000_21989AK0030001.txt,plan_0000_21989AK0030001_chunk_1,g.
2,plan_0001_21989AK0050001.txt,plan_0001_21989AK0050001_chunk_0,Plan Name: Delta Dental PPO 1000 Plan\nPlan ID: 21989AK0050001\nIssuer: Delta Dental of Alaska\nState: AK\nMetal Level: High\nPlan Type: PPO\nHSA ...
3,plan_0002_21989AK0050002.txt,plan_0002_21989AK0050002_chunk_0,Plan Name: Delta Dental PPO 1500 Plan\nPlan ID: 21989AK0050002\nIssuer: Delta Dental of Alaska\nState: AK\nMetal Level: High\nPlan Type: PPO\nHSA ...
4,plan_0003_21989AK0070001.txt,plan_0003_21989AK0070001_chunk_0,Plan Name: Delta Dental Premier Healthy Smiles\nPlan ID: 21989AK0070001\nIssuer: Delta Dental of Alaska\nState: AK\nMetal Level: Low\nPlan Type: I...


In [11]:
# Save chunks for reuse

chunks_output_path = PREPROCESSED_DIR / "rag_plan_chunks.csv"

chunks_df.to_csv(chunks_output_path, index=False)

print("Saved chunks to:", chunks_output_path)

Saved chunks to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\02_preprocessed\rag_plan_chunks.csv


In [12]:
# Load embedding model
# This model is lightweight and good for local semantic retrieval.

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [13]:
# Create embeddings for plan chunks

texts = chunks_df["text"].tolist()

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)

embeddings = np.array(embeddings).astype("float32")

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Embeddings shape: (1391, 384)


In [14]:
# Build FAISS vector index
# IndexFlatIP is appropriate because embeddings are normalized.

embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 1391


In [15]:
# Save FAISS index and chunk metadata

index_path = VECTOR_DB_DIR / "cms_plan_rag.index"
metadata_path = VECTOR_DB_DIR / "cms_plan_chunks_metadata.pkl"

faiss.write_index(index, str(index_path))

with open(metadata_path, "wb") as f:
    pickle.dump(chunks_df.to_dict("records"), f)

print("Saved index:", index_path)
print("Saved metadata:", metadata_path)

Saved index: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\vector_db\cms_plan_rag.index
Saved metadata: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\vector_db\cms_plan_chunks_metadata.pkl


In [16]:
# Create retrieval function

def retrieve_relevant_chunks(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
    )

    query_embedding = np.array(query_embedding).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        record = chunks_df.iloc[idx].to_dict()
        record["score"] = float(score)
        results.append(record)

    return results

In [17]:
# Test retrieval with an out-of-pocket question

query = "What is the out-of-pocket maximum for this plan?"

results = retrieve_relevant_chunks(query, top_k=5)

for result in results:
    print("\n---")
    print("Score:", round(result["score"], 4))
    print("Document:", result["document_name"])
    print(result["text"])


---
Score: 0.5741
Document: plan_0632_30115FL0020001.txt
Plan Name: BlueDental Choice Q
Plan ID: 30115FL0020001
Issuer: Florida Combined Life
State: FL
Metal Level: High
Plan Type: PPO
HSA Eligible: Not available

Deductible Design:
Medical and drug deductibles integrated: Not available.

Out-of-Pocket Design:
Medical and drug maximum out-of-pocket integrated: Not available.
The individual in-network tier 1 maximum out-of-pocket amount is $450.

Benefits Context:
This plan record comes from public CMS Exchange Public Use File data.
It can be used for benefits search, plan comparison, document retrieval, and RAG question answering.

---
Score: 0.5739
Document: plan_0451_67775DE0010004.txt
Plan Name: Select Plan Basic
Plan ID: 67775DE0010004
Issuer: Dominion National
State: DE
Metal Level: Low
Plan Type: HMO
HSA Eligible: Not available

Deductible Design:
Medical and drug deductibles integrated: Not available.

Out-of-Pocket Design:
Medical and drug maximum out-of-pocket integrated: Not

In [18]:
# Test retrieval with an HSA question

query = "Is this plan HSA eligible?"

results = retrieve_relevant_chunks(query, top_k=5)

for result in results:
    print("\n---")
    print("Score:", round(result["score"], 4))
    print("Document:", result["document_name"])
    print(result["text"])


---
Score: 0.6373
Document: plan_0181_70525AR0070285.txt
Plan Name: Choice Bronze HSA (QualChoice)
Plan ID: 70525AR0070285
Issuer: Ambetter from Arkansas Health & Wellness
State: AR
Metal Level: Expanded Bronze
Plan Type: POS
HSA Eligible: Yes

Deductible Design:
Medical and drug deductibles integrated: Yes.

Out-of-Pocket Design:
Medical and drug maximum out-of-pocket integrated: Yes.
The individual in-network tier 1 maximum out-of-pocket amount is Not available.

Benefits Context:
This plan record comes from public CMS Exchange Public Use File data.
It can be used for benefits search, plan comparison, document retrieval, and RAG question answering.

---
Score: 0.6281
Document: plan_0195_75293AR1200035.txt
Plan Name: Catastrophic HSA
Plan ID: 75293AR1200035
Issuer: Arkansas Blue Cross and Blue Shield
State: AR
Metal Level: Catastrophic
Plan Type: PPO
HSA Eligible: Yes

Deductible Design:
Medical and drug deductibles integrated: Yes.

Out-of-Pocket Design:
Medical and drug maximum out

In [19]:
# Test retrieval with a deductible design question

query = "Are medical and drug deductibles integrated?"

results = retrieve_relevant_chunks(query, top_k=5)

for result in results:
    print("\n---")
    print("Score:", round(result["score"], 4))
    print("Document:", result["document_name"])
    print(result["text"])


---
Score: 0.4891
Document: plan_0374_68445AZ0600010.txt
Plan Name: Bronze Complete 4 $0 Tier-1 PCP Visits, $0 Antidote 24/7 Virtual PCP/Urg/Chronic Care, $0 Core Rx
Plan ID: 68445AZ0600010
Issuer: Antidote Health Plan of Arizona, Inc.
State: AZ
Metal Level: Expanded Bronze
Plan Type: HMO
HSA Eligible: Yes

Deductible Design:
Medical and drug deductibles integrated: Yes.

Out-of-Pocket Design:
Medical and drug maximum out-of-pocket integrated: Yes.
The individual in-network tier 1 maximum out-of-pocket amount is Not available.

Benefits Context:
This plan record comes from public CMS Exchange Public Use File data.
It can be used for benefits search, plan comparison, document retrieval, and RAG question answering.

---
Score: 0.4859
Document: plan_0416_97667AZ0110016.txt
Plan Name: Connect Bronze 9500 Indiv Med Deductible
Plan ID: 97667AZ0110016
Issuer: Cigna HealthCare of Arizona, Inc
State: AZ
Metal Level: Expanded Bronze
Plan Type: HMO
HSA Eligible: Yes

Deductible Design:
Medical a

In [20]:
# Create source-backed answer function without an LLM
# This gives a transparent retrieval-only baseline before adding Gemini/Vertex AI.

def answer_question_without_llm(query, top_k=3):
    results = retrieve_relevant_chunks(query, top_k=top_k)

    answer = {
        "question": query,
        "answer": "The most relevant CMS plan records are listed below. Review the retrieved sources for exact plan details.",
        "sources": results,
    }

    return answer

In [21]:
# Test source-backed answer

response = answer_question_without_llm(
    "Which plans have an out-of-pocket maximum?",
    top_k=3,
)

print("Question:", response["question"])
print("Answer:", response["answer"])

for source in response["sources"]:
    print("\n--- Source ---")
    print("Score:", round(source["score"], 4))
    print("Document:", source["document_name"])
    print(source["text"])

Question: Which plans have an out-of-pocket maximum?
Answer: The most relevant CMS plan records are listed below. Review the retrieved sources for exact plan details.

--- Source ---
Score: 0.5584
Document: plan_0145_26904AR0020007.txt
Plan Name: BEST Life Preferred Dental Plan
Plan ID: 26904AR0020007
Issuer: BEST Life
State: AR
Metal Level: High
Plan Type: PPO
HSA Eligible: Not available

Deductible Design:
Medical and drug deductibles integrated: Not available.

Out-of-Pocket Design:
Medical and drug maximum out-of-pocket integrated: Not available.
The individual in-network tier 1 maximum out-of-pocket amount is $450.

Benefits Context:
This plan record comes from public CMS Exchange Public Use File data.
It can be used for benefits search, plan comparison, document retrieval, and RAG question answering.

--- Source ---
Score: 0.552
Document: plan_0043_12538AL0020007.txt
Plan Name: BEST Life Preferred Dental Plan
Plan ID: 12538AL0020007
Issuer: BEST Life
State: AL
Metal Level: High
P

In [22]:
# Create cleaner display table for retrieval results

def retrieval_results_to_table(query, top_k=5):
    results = retrieve_relevant_chunks(query, top_k=top_k)
    results_df = pd.DataFrame(results)

    return results_df[
        [
            "score",
            "document_name",
            "chunk_id",
            "text",
        ]
    ]


retrieval_table = retrieval_results_to_table(
    "What is the out-of-pocket maximum?",
    top_k=5,
)

display(retrieval_table)

,score,document_name,chunk_id,text
0,0.445942,plan_0146_26904AR0020008.txt,plan_0146_26904AR0020008_chunk_0,Plan Name: BEST Life Essential Basic Dental Plan\nPlan ID: 26904AR0020008\nIssuer: BEST Life\nState: AR\nMetal Level: High\nPlan Type: PPO\nHSA El...
1,0.441957,plan_0459_67775DE0020008.txt,plan_0459_67775DE0020008_chunk_0,Plan Name: Elite PPO Preventive\nPlan ID: 67775DE0020008\nIssuer: Dominion National\nState: DE\nMetal Level: Low\nPlan Type: PPO\nHSA Eligible: No...
2,0.441617,plan_0145_26904AR0020007.txt,plan_0145_26904AR0020007_chunk_0,Plan Name: BEST Life Preferred Dental Plan\nPlan ID: 26904AR0020007\nIssuer: BEST Life\nState: AR\nMetal Level: High\nPlan Type: PPO\nHSA Eligible...
3,0.440851,plan_0144_26904AR0020006.txt,plan_0144_26904AR0020006_chunk_0,Plan Name: BEST Life Essential Value Dental Plan\nPlan ID: 26904AR0020006\nIssuer: BEST Life\nState: AR\nMetal Level: High\nPlan Type: PPO\nHSA El...
4,0.439731,plan_0044_12538AL0020008.txt,plan_0044_12538AL0020008_chunk_0,Plan Name: BEST Life Essential Basic Dental Plan\nPlan ID: 12538AL0020008\nIssuer: BEST Life\nState: AL\nMetal Level: Low\nPlan Type: PPO\nHSA El...


In [23]:
# Save sample retrieval result for reporting and Notebook 5 governance logs

sample_query = "What is the out-of-pocket maximum for a plan?"
sample_results = retrieve_relevant_chunks(sample_query, top_k=5)

sample_results_df = pd.DataFrame(sample_results)

sample_output_path = PREPROCESSED_DIR / "sample_rag_retrieval_results.csv"
sample_results_df.to_csv(sample_output_path, index=False)

print("Saved sample retrieval results to:", sample_output_path)
display(sample_results_df)

Saved sample retrieval results to: c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\02_preprocessed\sample_rag_retrieval_results.csv


,document_name,chunk_id,text,score
0,plan_0451_67775DE0010004.txt,plan_0451_67775DE0010004_chunk_0,Plan Name: Select Plan Basic\nPlan ID: 67775DE0010004\nIssuer: Dominion National\nState: DE\nMetal Level: Low\nPlan Type: HMO\nHSA Eligible: Not a...,0.566556
1,plan_0917_74243FL0010003.txt,plan_0917_74243FL0010003_chunk_0,Plan Name: Choice PPO Plus\nPlan ID: 74243FL0010003\nIssuer: Dominion National\nState: FL\nMetal Level: Low\nPlan Type: PPO\nHSA Eligible: Not ava...,0.565149
2,plan_0915_74243FL0010001.txt,plan_0915_74243FL0010001_chunk_0,Plan Name: Choice PPO Basic\nPlan ID: 74243FL0010001\nIssuer: Dominion National\nState: FL\nMetal Level: Low\nPlan Type: PPO\nHSA Eligible: Not av...,0.564147
3,plan_0918_74243FL0010004.txt,plan_0918_74243FL0010004_chunk_0,Plan Name: Choice PPO Preventive\nPlan ID: 74243FL0010004\nIssuer: Dominion National\nState: FL\nMetal Level: Low\nPlan Type: PPO\nHSA Eligible: N...,0.561745
4,plan_0916_74243FL0010002.txt,plan_0916_74243FL0010002_chunk_0,Plan Name: Choice PPO Premium\nPlan ID: 74243FL0010002\nIssuer: Dominion National\nState: FL\nMetal Level: High\nPlan Type: PPO\nHSA Eligible: Not...,0.560491


In [24]:
# Confirm required files exist for Notebook 5

required_outputs = [
    PREPROCESSED_DIR / "rag_plan_chunks.csv",
    PREPROCESSED_DIR / "sample_rag_retrieval_results.csv",
    VECTOR_DB_DIR / "cms_plan_rag.index",
    VECTOR_DB_DIR / "cms_plan_chunks_metadata.pkl",
]

for file_path in required_outputs:
    print(file_path, "exists:", file_path.exists())

c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\02_preprocessed\rag_plan_chunks.csv exists: True
c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\data\02_preprocessed\sample_rag_retrieval_results.csv exists: True
c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\vector_db\cms_plan_rag.index exists: True
c:\Users\tevin\OneDrive\Desktop\LLM-And-Generative-AI\rag\benefits_ai_platform\vector_db\cms_plan_chunks_metadata.pkl exists: True


In [25]:
# Final notebook summary

print("Notebook 4 RAG prototype complete.")
print("Plan records used:", len(plan_df))
print("Documents created:", len(text_files))
print("Chunks created:", len(chunks_df))
print("Vector index size:", index.ntotal)

print("\nFiles created:")
print("1. data/02_preprocessed/rag_documents/")
print("2. data/02_preprocessed/rag_plan_chunks.csv")
print("3. vector_db/cms_plan_rag.index")
print("4. vector_db/cms_plan_chunks_metadata.pkl")
print("5. data/02_preprocessed/sample_rag_retrieval_results.csv")

Notebook 4 RAG prototype complete.
Plan records used: 1000
Documents created: 1000
Chunks created: 1391
Vector index size: 1391

Files created:
1. data/02_preprocessed/rag_documents/
2. data/02_preprocessed/rag_plan_chunks.csv
3. vector_db/cms_plan_rag.index
4. vector_db/cms_plan_chunks_metadata.pkl
5. data/02_preprocessed/sample_rag_retrieval_results.csv
